# GCM BA - 6. Ad hoc test: the jitter of the dry days of pr

**Ad hoc test, outside the F1 chain**: it reads the inputs of notebook 2, the F1 products of notebook 3 (tasmax) and
the statistics cached by notebook 5, and writes only in `GCM_BA/6_test_jitter/` (EOS) and `outputs/` next to it.

**Question.** F1 (5_dependence) weakens the summer correlation between daily pr and tasmax (JJA Spearman: ERA5
−0.330, raw −0.279, F1 −0.237). Likely cause, the jitter: every pr value below 0.1 mm/day is replaced by a uniform
noise in (0, 0.1), in ERA5, in EC-Earth3 historical and in the runs, before the multiplicative QDM on all days
(Cannon et al. 2015). In summer EC-Earth3 has too many dry days (August 53 % against 41 % in ERA5 at 0.1 mm/day), so
QDM turns the dry days of highest *noise* into small wet days: they are chosen at random, hot days as often as cool
ones, which dilutes the pr–tasmax relation.

**Variant.** The same F1 code (`bu.fit_F`, `bu.apply_F`, xsdba `jitter_under_thresh`), pr only, with only the jitter
threshold changed: values below `JIT` (0.001 mm/day) are jittered in (0, `JIT`), the model's own trace amounts
between `JIT` and 0.1 mm/day are kept, so they rank the dry days: the "dampest" dry days of the model (often next to a
rainy day, under clouds) become wet first. Values below 0.1 mm/day are set to 0 after the QDM, as in F1. Section 1
checks that the model and ERA5 do have such trace amounts (if most dry days were exact zeros, the variant would
change nothing).

**Expected.** Calibration unchanged (QDM maps the ranks: the dry-day fraction and the quantiles stay those of ERA5);
the days of 1 mm or more are the model's own wet days in both versions (the noise only reorders the days below
0.1 mm), so the statistics of 5_dependence that use wet = pr ≥ 1 mm/day (transitions, spells, CDD, areal extremes)
should barely change; the pr–tasmax correlation, which uses all the days, should get closer to the raw model.
This test does **not** address the damped dry-spell change and the amplified areal Rx1day change of the hosing runs:
they come from the quantile mapping itself (part of the frequency change moved to the intensity), another test.

**Decision rule.** The variant replaces the present jitter for F2 if (1) the calibration against ERA5 is unchanged,
(2) the JJA pr–tasmax correlation is closer to the raw model (−0.279) and to ERA5, (3) the changes of the dependence
(climate change, hosing) are not worse.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, json, time, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
import validation_utils as vu
warnings.filterwarnings("ignore")

CONFIG_PATH = None
cfg = bu.load_config(CONFIG_PATH)
CAL_PERIOD = ("1981", "2014")      # as F1
REGION = bu.gcm_ba_region(cfg, 1.0)
F1_TAG = f"F1_{CAL_PERIOD[0]}-{CAL_PERIOD[1]}_{bu.F1_VARIANT}" + ("_europe" if REGION else "")
JIT = 0.001                        # mm/day: only the values below it are jittered (F1: bu.THRESH_PR = 0.1)
TEST_TAG = f"{F1_TAG}_jit{JIT:g}"
CHUNKS_SPACE = {"lat": 32, "lon": 64}
INTERP = "linear"                  # as notebook 3
GCM_RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
RUNS = ["gcm_historical"] + GCM_RUNS
OVERWRITE = False                  # False: parameters, products and statistics already written are read back

DOMAIN = vu.init(cfg, "domain", borders_res="01M")        # LISFLOOD domain without the land outside Europe (as 5)
OUTF1 = bu.gcm_ba_dir(cfg, "GCM_bias_adjusted_0.7deg")    # F1 products (notebook 3): tasmax, and pr of the present F1
CACHE5 = bu.gcm_ba_dir(cfg, "5_dependence", "cache" + ("_europe" if REGION else ""))   # statistics of 5 (read only)
TEST = bu.gcm_ba_dir(cfg, "6_test_jitter", TEST_TAG)      # the only folders written on EOS
FDIR, PDIR, CACHE = TEST / "params", TEST / "products", TEST / "cache"
for d in (FDIR, PDIR, CACHE):
    d.mkdir(parents=True, exist_ok=True)
OUT = Path.cwd() / "outputs"; OUT.mkdir(exist_ok=True)
LABEL = {"gcm_historical": "historical", "gcm_2025_nohosing": "2025 original", "gcm_2025_hosing": "2025 hosing",
         "gcm_2050_nohosing": "2050 original", "gcm_2050_hosing": "2050 hosing"}
PAIRS = [("gcm_2025_nohosing", "gcm_historical", "2025 original - historical"),
         ("gcm_2050_nohosing", "gcm_historical", "2050 original - historical"),
         ("gcm_2025_hosing", "gcm_2025_nohosing", "2025 hosing - original"),
         ("gcm_2050_hosing", "gcm_2050_nohosing", "2050 hosing - original")]
period_of = lambda name: CAL_PERIOD if name in ("era5", "gcm_historical") else None
print(f"F1 {F1_TAG} | test {TEST_TAG} | jitter below {JIT} mm/day (F1: {bu.THRESH_PR}) | written in {TEST}")

## 1. The dry days: exact zeros or trace amounts?

Share of the days below 0.1 mm/day, and among them the shares of exact zeros, of values in (0, `JIT`) (still jittered
by the variant) and of trace amounts in [`JIT`, 0.1) (kept by the variant, they rank the dry days), domain pixels.

In [ ]:
def pr_pixels(name):
    return vu.domain_pixels(bu.gcm_ba_input(cfg, name, "pr", period_of(name), REGION))


rows = []
for name in ("era5", "gcm_historical", "gcm_2050_nohosing", "gcm_2050_hosing"):
    p = pr_pixels(name)
    x, mon = p["x"], p["month"]
    for s in ("DJF", "JJA"):
        v = x[np.isin(mon, vu.SEASONS[s])]
        v = v[np.isfinite(v)]
        dry = v[v < bu.THRESH_PR]
        rows.append({"dataset": name, "season": s, "dry days (< 0.1 mm, %)": 100 * dry.size / v.size,
                     "of which exact 0 (%)": 100 * np.mean(dry == 0),
                     f"of which (0, {JIT:g}) (%)": 100 * np.mean((dry > 0) & (dry < JIT)),
                     f"of which [{JIT:g}, 0.1) (%)": 100 * np.mean(dry >= JIT)})
    del p, x
DRY = pd.DataFrame(rows).set_index(["dataset", "season"])
DRY.to_csv(OUT / f"dry_days_{TEST_TAG}.csv")
display(DRY.round(1))

## 2. F1 with the new jitter (pr only): calibration on 1981–2014, application to the historical run and the four runs

The F1 code of notebooks 2 and 3, unchanged, with `bu._jitter_dry` replaced for this notebook only (the module is not
modified on disk): `jitter_under_thresh(x, JIT)` instead of `jitter_under_thresh(x, 0.1)`. Same seeds, windows,
quantiles, interpolation, tail rule and European mask as F1.

In [ ]:
def _jitter_small(x, seed):
    # as bu._jitter_dry, with the threshold JIT instead of bu.THRESH_PR
    import dask.array as dsa
    from xsdba.processing import jitter_under_thresh
    x = x.copy()
    x.attrs["units"] = "mm/d"
    dsa.random.seed(seed)
    return jitter_under_thresh(x, f"{JIT} mm/d")


bu._jitter_dry = _jitter_small                      # this notebook only
t0 = time.time()
ref = {"pr": bu.gcm_ba_input(cfg, "era5", "pr", CAL_PERIOD, REGION)}
hist = {"pr": bu.gcm_ba_input(cfg, "gcm_historical", "pr", CAL_PERIOD, REGION)}
bu.fit_F(ref, hist, FDIR, chunks_space=CHUNKS_SPACE, overwrite=OVERWRITE)
print(f"calibration: {(time.time() - t0) / 60:.1f} min")
for name in RUNS:
    path = PDIR / f"{name}_{TEST_TAG}.nc"
    if path.exists() and not OVERWRITE:
        print(f"[already done] {name}"); continue
    t0 = time.time()
    adj = bu.apply_F({"pr": bu.gcm_ba_input(cfg, name, "pr", period_of(name), REGION)}, FDIR,
                     chunks_space=CHUNKS_SPACE, interp=INTERP, pr_tail=None)
    adj = {v: a.where(bu.keep_mask(a)) for v, a in adj.items()}
    bu.save_ds(xr.Dataset(adj), path)
    print(f"{name}: {(time.time() - t0) / 60:.1f} min", flush=True)

## 3. Calibration against ERA5 and AMOC signal on the mean (domain pixels)

Historical 1981–2014: mean pr, dry-day fraction (< 0.1 mm), wet-day fraction (≥ 1 mm), mean of the annual maxima,
per season, for ERA5, raw, F1 and the variant. Runs: change of the mean pr (%) with the hosing.

In [ ]:
def pr_of(name, kind):
    # pr of a dataset on the domain pixels: kind "raw" (input), "F1" (notebook 3), "jit" (this notebook)
    if kind == "raw":
        return pr_pixels(name)
    f = (OUTF1 / f"{name}_{F1_TAG}.nc") if kind == "F1" else (PDIR / f"{name}_{TEST_TAG}.nc")
    da = xr.open_dataset(f, chunks={"time": 3100}, **bu._TIME_KW)["pr"]
    return vu.domain_pixels(da if period_of(name) is None else bu.subset_time(da, *period_of(name)))


def calib(p):
    x, mon, yr = p["x"], p["month"], p["year"]
    out = {}
    for s in ("DJF", "JJA"):
        v = x[np.isin(mon, vu.SEASONS[s])]
        out[f"mean pr {s} (mm/day)"] = float(np.nanmean(v))
        out[f"dry days {s} (< 0.1 mm, %)"] = 100 * float(np.nanmean(v < bu.THRESH_PR))
        out[f"wet days {s} (>= 1 mm, %)"] = 100 * float(np.nanmean(v >= vu.WET_DEP))
    rx = np.stack([np.nanmax(x[yr == y], axis=0) for y in np.unique(yr)])
    out["Rx1day, mean of the pixels (mm/day)"] = float(np.nanmean(rx))
    return out


CAL = {}
for name, kinds in [("era5", ("raw",))] + [(n, ("raw", "F1", "jit")) for n in RUNS]:
    for k in kinds:
        key = f"{name} {k}"
        path = CACHE / f"calib_{name}_{k}.pkl"
        if path.exists() and not OVERWRITE:
            CAL[key] = pd.read_pickle(path); continue
        p = pr_of(name, k)
        CAL[key] = calib(p); pd.to_pickle(CAL[key], path)
        del p
CALT = pd.DataFrame(CAL).T
CALT.to_csv(OUT / f"calibration_{TEST_TAG}.csv")
show = CALT.loc[["era5 raw", "gcm_historical raw", "gcm_historical F1", "gcm_historical jit"]]
print("historical 1981-2014 against ERA5"); display(show.round(3))
rows = []
for h, o in (("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")):
    for k in ("raw", "F1", "jit"):
        rows.append({"pair": f"{h} - {o}", "version": k,
                     **{c: 100 * (CAL[f"{h} {k}"][c] / CAL[f"{o} {k}"][c] - 1) for c in CALT.columns if c.startswith("mean pr")}})
print("hosing - original, change of the mean pr (%)"); display(pd.DataFrame(rows).set_index(["pair", "version"]).round(2))

## 4. Dependence: the statistics of 5_dependence for the variant

ERA5, raw and F1 from the cache of notebook 5 (computed here, in this notebook's cache, if missing); the variant with
its pr and the tasmax of F1 (tasmax is not changed by the jitter).

In [ ]:
def series(name, kind):
    pr = pr_of(name, kind)
    if kind == "raw":
        tx = vu.domain_pixels(bu.gcm_ba_input(cfg, name, "tasmax", period_of(name), REGION))
    else:
        da = xr.open_dataset(OUTF1 / f"{name}_{F1_TAG}.nc", chunks={"time": 3100}, **bu._TIME_KW)["tasmax"]
        tx = vu.domain_pixels(da if period_of(name) is None else bu.subset_time(da, *period_of(name)))
    return pr, tx


DEP = {}
for name, kind, key in [("era5", "raw", "ERA5")] + [(n, k, f"{l} {k}") for n, l in LABEL.items() for k in ("raw", "F1", "jit")]:
    tag = {"raw": "raw", "F1": F1_TAG, "jit": TEST_TAG}[kind]
    p5, p6 = CACHE5 / f"{name}_{tag}_dependence.pkl", CACHE / f"{name}_{tag}_dependence.pkl"
    if kind != "jit" and p5.exists():
        DEP[key] = pd.read_pickle(p5); continue
    if p6.exists() and not OVERWRITE:
        DEP[key] = pd.read_pickle(p6); continue
    pr, tx = series(name, kind)
    DEP[key] = vu.dependence_stats(pr, tx, p6, True, label=key)
    del pr, tx
W = np.cos(np.deg2rad(DEP["ERA5"]["lat"]))
wmean = lambda a: float(np.nansum(W * a) / np.sum(W * np.isfinite(a)))

E, R, A, J = (DEP[k]["temporal"] for k in ("ERA5", "historical raw", "historical F1", "historical jit"))
rows = []
for c in E.columns:
    e, r, a, j = (d[c].values for d in (E, R, A, J))
    ok = np.isfinite(a) & np.isfinite(j)
    rows.append({"statistic": c, "ERA5": wmean(e), "raw": wmean(r), "F1": wmean(a), "jit": wmean(j),
                 "jit vs F1: pattern corr": np.corrcoef(a[ok], j[ok])[0, 1], "|bias| F1": abs(wmean(a - e)), "|bias| jit": abs(wmean(j - e))})
temporal = pd.DataFrame(rows).set_index("statistic")
temporal.to_csv(OUT / f"dependence_temporal_{TEST_TAG}.csv")
spatial = pd.DataFrame({k: {f"{v}, {s}": vu.efold_km(d.index.values, d[(v, s)].values) for (v, s) in d.columns}
                        for k, d in ((k, DEP[k]["spatial"]) for k in ("ERA5", "historical raw", "historical F1", "historical jit"))})
spatial.index.name = "e-folding distance (km)"
rows = []
for k in vu.DEP_BLOCKS:
    for stat in ("Rx1day", "Rx5day", "ARF"):
        if k == 1 and stat == "ARF":
            continue
        rows.append({"block": f"{k} x {k} cells", "statistic": stat,
                     **{n: float(np.median(DEP[n]["areal"][k][stat])) for n in ("ERA5", "historical raw", "historical F1", "historical jit")}})
areal = pd.DataFrame(rows).set_index(["block", "statistic"])
with pd.option_context("display.width", 250, "display.max_columns", 20):
    print("(1) historical, temporal and between variables (domain means)"); display(temporal.round(3))
    print("(2) e-folding distance (km)"); display(spatial.round(0))
    print("(3) areal extremes (median over the blocks)"); display(areal.round(3))

## 5. Changes of the dependence: raw, F1 and the variant

As section 3 of notebook 5: domain-mean change of each statistic and pattern regression of the adjusted change on the
raw change (slope 1 and correlation 1 = preserved), for F1 and for the variant.

In [ ]:
def slope_corr(x, y):
    ok = np.isfinite(x) & np.isfinite(y)
    w = W[ok]; x, y = x[ok], y[ok]
    mx, my = np.average(x, weights=w), np.average(y, weights=w)
    vx, vy = np.average((x - mx) ** 2, weights=w), np.average((y - my) ** 2, weights=w)
    cov = np.average((x - mx) * (y - my), weights=w)
    return (cov / vx if vx > 0 else np.nan), (cov / np.sqrt(vx * vy) if vx > 0 and vy > 0 else np.nan)


rows = []
for a_, b_, lab in PAIRS:
    ka, kb = LABEL[a_], LABEL[b_]
    if any(f"{k} {t}" not in DEP for k in (ka, kb) for t in ("raw", "F1", "jit")):
        continue
    for c in E.columns:
        d = {t: DEP[f"{ka} {t}"]["temporal"][c].values - DEP[f"{kb} {t}"]["temporal"][c].values for t in ("raw", "F1", "jit")}
        sF, rF = slope_corr(d["raw"], d["F1"]); sJ, rJ = slope_corr(d["raw"], d["jit"])
        rows.append({"pair": lab, "statistic": c, "change raw": wmean(d["raw"]), "change F1": wmean(d["F1"]), "change jit": wmean(d["jit"]),
                     "slope F1": sF, "slope jit": sJ, "corr F1": rF, "corr jit": rJ})
    for k in vu.DEP_BLOCKS[1:]:
        for stat in ("Rx1day", "Rx5day"):
            ch = {t: 100 * (np.median(DEP[f"{ka} {t}"]["areal"][k][stat]) / np.median(DEP[f"{kb} {t}"]["areal"][k][stat]) - 1)
                  for t in ("raw", "F1", "jit")}
            rows.append({"pair": lab, "statistic": f"areal {stat}, {k} x {k} cells (%)", **{f"change {t}": v for t, v in ch.items()}})
signals = pd.DataFrame(rows).set_index(["pair", "statistic"])
signals.to_csv(OUT / f"dependence_signals_{TEST_TAG}.csv")
key = ["Spearman pr - tasmax, JJA", "Spearman pr - tasmax, DJF", "P(wet | wet), JJA", "mean dry spell (days)", "CDD (days)",
       "areal Rx1day, 3 x 3 cells (%)", "areal Rx1day, 6 x 6 cells (%)"]
with pd.option_context("display.width", 250, "display.max_rows", 200):
    print("key statistics"); display(signals[signals.index.get_level_values("statistic").isin(key)].round(3))
    print("all"); display(signals.round(3))

## 6. Reading

- **Section 1**: if the dry days of EC-Earth3 are mostly exact zeros, the variant cannot rank them and changes little.
- **Calibration** (section 3): the variant must stay as close to ERA5 as F1 (dry-day fraction, wet days, mean,
  Rx1day) and keep the hosing change of the mean pr.
- **pr–tasmax** (section 4): JJA Spearman of the variant against F1 (−0.237), raw (−0.279) and ERA5 (−0.330); the
  pattern correlation jit vs F1 shows where it changes.
- **Changes** (section 5): slopes of the variant against those of F1; the dry spells and areal Rx1day are expected
  to stay as in F1 (wet days of 1 mm or more are the model's own in both).